In [2]:
"""
=============================================================================
CLASSIFICAÇÃO BINÁRIA DE LIBERAÇÃO DE EMPRÉSTIMOS
=============================================================================

Dataset: loan.csv
Objetivo: Prever se um empréstimo será aprovado ou não (loan_status)

Etapas:
a) Split treino/teste
b) Balanceamento com SMOTE (apenas no treino)
c) Remoção de colunas específicas
d) Tratamento de valores faltantes
e) Encoding de variáveis categóricas
f) Tratamento especial da coluna 'dependents'
g) Padronização de features numéricas
h) 5 Pipelines com Grid Search e validação cruzada
i) Seleção e salvamento do melhor modelo
j) Deploy com Streamlit

=============================================================================
"""

# =============================================================================
# 1. IMPORTAÇÕES
# =============================================================================
print("=" * 80)
print("IMPORTANDO BIBLIOTECAS")
print("=" * 80)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

# Métricas
from sklearn.metrics import (classification_report, roc_auc_score, 
                              roc_curve, confusion_matrix, ConfusionMatrixDisplay)

# SMOTE
from imblearn.over_sampling import SMOTE

# Persistência
import pickle
import joblib

print("✓ Bibliotecas importadas com sucesso!\n")


IMPORTANDO BIBLIOTECAS
✓ Bibliotecas importadas com sucesso!



In [3]:
# =============================================================================
# 2. CARREGAMENTO DOS DADOS
# =============================================================================
print("=" * 80)
print("CARREGANDO DATASET")
print("=" * 80)

# URL do dataset
url = "https://raw.githubusercontent.com/josenalde/machinelearning/main/src/dataset/loan.csv"

# Carregando
df = pd.read_csv(url)

print(f"✓ Dataset carregado com sucesso!")
print(f"  Shape: {df.shape}")
print(f"  Colunas: {list(df.columns)}\n")

# Visualizando as primeiras linhas
print("Primeiras linhas do dataset:")
print(df.head())
print()

# Informações básicas
print("Informações do dataset:")
print(df.info())
print()


CARREGANDO DATASET
✓ Dataset carregado com sucesso!
  Shape: (614, 13)
  Colunas: ['Loan_ID', 'Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History', 'Property_Area', 'Loan_Status']

Primeiras linhas do dataset:
    Loan_ID Gender Married Dependents     Education Self_Employed  \
0  LP001002   Male      No          0      Graduate            No   
1  LP001003   Male     Yes          1      Graduate            No   
2  LP001005   Male     Yes          0      Graduate           Yes   
3  LP001006   Male     Yes          0  Not Graduate            No   
4  LP001008   Male      No          0      Graduate            No   

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0         NaN             360.0   
1             4583             1508.0       128.0             360.0   
2             3000                0.0        66.0          

In [5]:
# =============================================================================
# 3. ANÁLISE EXPLORATÓRIA INICIAL
# =============================================================================
print("=" * 80)
print("ANÁLISE EXPLORATÓRIA")
print("=" * 80)

print("\n Estatísticas descritivas:")
print(df.describe())
print()

print(" Valores únicos por coluna:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()} valores únicos")
print()

print(" Valores faltantes:")
print(df.isnull().sum())
print(f"\nTotal de valores faltantes: {df.isnull().sum().sum()}")
print()

# Verificando balanceamento da variável alvo
print("=" * 80)
print("BALANCEAMENTO DA VARIÁVEL ALVO (Loan_Status)")
print("=" * 80)

print(df['Loan_Status'].value_counts())
print("\nProporção:")
print(df['Loan_Status'].value_counts(normalize=True))
print()


ANÁLISE EXPLORATÓRIA

 Estatísticas descritivas:
       ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
count       614.000000         614.000000  592.000000         600.00000   
mean       5403.459283        1621.245798  146.412162         342.00000   
std        6109.041673        2926.248369   85.587325          65.12041   
min         150.000000           0.000000    9.000000          12.00000   
25%        2877.500000           0.000000  100.000000         360.00000   
50%        3812.500000        1188.500000  128.000000         360.00000   
75%        5795.000000        2297.250000  168.000000         360.00000   
max       81000.000000       41667.000000  700.000000         480.00000   

       Credit_History  
count      564.000000  
mean         0.842199  
std          0.364878  
min          0.000000  
25%          1.000000  
50%          1.000000  
75%          1.000000  
max          1.000000  

 Valores únicos por coluna:
  Loan_ID: 614 valores únicos


In [6]:
# =============================================================================
# 4. SEPARAÇÃO TREINO/TESTE (ANTES DE QUALQUER PROCESSAMENTO)
# =============================================================================
print("=" * 80)
print("ETAPA A: SEPARAÇÃO TREINO/TESTE")
print("=" * 80)

# Separando X e y
X = df.drop(columns=['Loan_Status'])
y = df['Loan_Status']

print(f"  Features (X): {X.shape}")
print(f"  Target (y): {y.shape}")

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✓ Split realizado (80/20):")
print(f"  Treino: {X_train.shape[0]} amostras")
print(f"  Teste: {X_test.shape[0]} amostras")
print(f"\nDistribuição no treino:")
print(y_train.value_counts())
print(f"\nDistribuição no teste:")
print(y_test.value_counts())
print()

ETAPA A: SEPARAÇÃO TREINO/TESTE
  Features (X): (614, 12)
  Target (y): (614,)

✓ Split realizado (80/20):
  Treino: 491 amostras
  Teste: 123 amostras

Distribuição no treino:
Loan_Status
Y    337
N    154
Name: count, dtype: int64

Distribuição no teste:
Loan_Status
Y    85
N    38
Name: count, dtype: int64



In [7]:
# =============================================================================
# 5. ETAPA B: REMOÇÃO DE COLUNAS ESPECIFICADAS
# =============================================================================
print("=" * 80)
print("ETAPA B (PARTE 1): REMOÇÃO DE COLUNAS")
print("=" * 80)

colunas_remover = ['Loan_ID', 'CoapplicantIncome', 'Loan_Amount_Term', 
                   'Credit_History', 'Property_Area']

print(f"Colunas a remover: {colunas_remover}")

# Remover das colunas (com verificação)
colunas_existentes = [col for col in colunas_remover if col in X_train.columns]
print(f"Colunas encontradas para remover: {colunas_existentes}")

X_train = X_train.drop(columns=colunas_existentes)
X_test = X_test.drop(columns=colunas_existentes)

print(f"\n✓ Colunas removidas!")
print(f"  Shape treino: {X_train.shape}")
print(f"  Shape teste: {X_test.shape}")
print(f"  Colunas restantes: {list(X_train.columns)}\n")


ETAPA B (PARTE 1): REMOÇÃO DE COLUNAS
Colunas a remover: ['Loan_ID', 'CoapplicantIncome', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']
Colunas encontradas para remover: ['Loan_ID', 'CoapplicantIncome', 'Loan_Amount_Term', 'Credit_History', 'Property_Area']

✓ Colunas removidas!
  Shape treino: (491, 7)
  Shape teste: (123, 7)
  Colunas restantes: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'LoanAmount']



In [8]:
# =============================================================================
# 6. ETAPA C: TRATAMENTO DE VALORES FALTANTES
# =============================================================================
print("=" * 80)
print("ETAPA C: TRATAMENTO DE VALORES FALTANTES")
print("=" * 80)

# Colunas categóricas a preencher com moda
colunas_categoricas_moda = ['Dependents', 'Self_Employed', 'Married', 'Gender']

print(f"Colunas categóricas a preencher com moda: {colunas_categoricas_moda}")
print()

# Verificar valores faltantes ANTES
print("Valores faltantes ANTES do tratamento:")
print(X_train.isnull().sum())
print()

# Preencher categóricas com moda
imputer_moda = SimpleImputer(strategy='most_frequent')

for col in colunas_categoricas_moda:
    if col in X_train.columns:
        if X_train[col].isnull().sum() > 0:
            X_train[col] = imputer_moda.fit_transform(X_train[[col]]).ravel()
            X_test[col] = imputer_moda.transform(X_test[[col]]).ravel()
            print(f"✓ Coluna '{col}' preenchida com moda")

print()

# Preencher numéricas com mediana
colunas_numericas = X_train.select_dtypes(include=[np.number]).columns
imputer_mediana = SimpleImputer(strategy='median')

for col in colunas_numericas:
    if X_train[col].isnull().sum() > 0:
        X_train[col] = imputer_mediana.fit_transform(X_train[[col]]).ravel()
        X_test[col] = imputer_mediana.transform(X_test[[col]]).ravel()
        print(f"✓ Coluna numérica '{col}' preenchida com mediana")

print()

# ETAPA D: Confirmação de que não há valores faltantes
print("=" * 80)
print("ETAPA D: CONFIRMAÇÃO - NÃO HÁ VALORES FALTANTES")
print("=" * 80)

print("Valores faltantes APÓS o tratamento:")
print(X_train.isnull().sum())
print(f"\n✓ Total de valores faltantes no treino: {X_train.isnull().sum().sum()}")
print(f"✓ Total de valores faltantes no teste: {X_test.isnull().sum().sum()}")
print()

ETAPA C: TRATAMENTO DE VALORES FALTANTES
Colunas categóricas a preencher com moda: ['Dependents', 'Self_Employed', 'Married', 'Gender']

Valores faltantes ANTES do tratamento:
Gender             11
Married             3
Dependents          8
Education           0
Self_Employed      27
ApplicantIncome     0
LoanAmount         20
dtype: int64

✓ Coluna 'Dependents' preenchida com moda
✓ Coluna 'Self_Employed' preenchida com moda
✓ Coluna 'Married' preenchida com moda
✓ Coluna 'Gender' preenchida com moda

✓ Coluna numérica 'LoanAmount' preenchida com mediana

ETAPA D: CONFIRMAÇÃO - NÃO HÁ VALORES FALTANTES
Valores faltantes APÓS o tratamento:
Gender             0
Married            0
Dependents         0
Education          0
Self_Employed      0
ApplicantIncome    0
LoanAmount         0
dtype: int64

✓ Total de valores faltantes no treino: 0
✓ Total de valores faltantes no teste: 0



In [9]:
# =============================================================================
# 7. ETAPA F: TRATAMENTO DA COLUNA 'DEPENDENTS'
# =============================================================================
print("=" * 80)
print("ETAPA F: TRATAMENTO DA COLUNA 'DEPENDENTS' (3+ → 3)")
print("=" * 80)

if 'Dependents' in X_train.columns:
    print(f"Valores únicos ANTES: {X_train['Dependents'].unique()}")
    
    # Substituir '3+' por '3'
    X_train['Dependents'] = X_train['Dependents'].replace('3+', '3')
    X_test['Dependents'] = X_test['Dependents'].replace('3+', '3')
    
    # Converter para int
    X_train['Dependents'] = X_train['Dependents'].astype(int)
    X_test['Dependents'] = X_test['Dependents'].astype(int)
    
    print(f"Valores únicos APÓS: {X_train['Dependents'].unique()}")
    print(f"✓ Coluna 'Dependents' tratada (3+ → 3 e convertida para int)\n")
else:
    print("⚠ Coluna 'Dependents' não encontrada\n")

ETAPA F: TRATAMENTO DA COLUNA 'DEPENDENTS' (3+ → 3)
Valores únicos ANTES: ['0' '1' '2' '3+']
Valores únicos APÓS: [0 1 2 3]
✓ Coluna 'Dependents' tratada (3+ → 3 e convertida para int)



In [10]:
# =============================================================================
# 8. ETAPA E: ENCODING DE VARIÁVEIS CATEGÓRICAS
# =============================================================================
print("=" * 80)
print("ETAPA E: ENCODING DE VARIÁVEIS CATEGÓRICAS DICOTÔMICAS")
print("=" * 80)

# Variáveis categóricas dicotômicas a encodar
colunas_encoder = ['Gender', 'Married', 'Education', 'Self_Employed']

print(f"Colunas para LabelEncoder: {colunas_encoder}")
print()

# Encoders para cada coluna
encoders = {}

for col in colunas_encoder:
    if col in X_train.columns:
        print(f"Encodando '{col}'...")
        print(f"  Valores únicos antes: {X_train[col].unique()}")
        
        # Criar e treinar encoder
        le = LabelEncoder()
        X_train[col] = le.fit_transform(X_train[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))
        
        # Salvar encoder
        encoders[col] = le
        
        print(f"  Valores após encoding: {X_train[col].unique()}")
        print(f"  Mapeamento: {dict(zip(le.classes_, le.transform(le.classes_)))}")
        print()

# Encoding da variável target
print("Encodando variável target (Loan_Status)...")
print(f"  Valores únicos antes: {y_train.unique()}")

le_target = LabelEncoder()
y_train_encoded = le_target.fit_transform(y_train.astype(str))
y_test_encoded = le_target.transform(y_test.astype(str))

print(f"  Valores após encoding: {np.unique(y_train_encoded)}")
print(f"  Mapeamento: {dict(zip(le_target.classes_, le_target.transform(le_target.classes_)))}")
print()

print(f"✓ Encoding concluído!")
print(f"  Encoders salvos: {list(encoders.keys())}")
print()


ETAPA E: ENCODING DE VARIÁVEIS CATEGÓRICAS DICOTÔMICAS
Colunas para LabelEncoder: ['Gender', 'Married', 'Education', 'Self_Employed']

Encodando 'Gender'...
  Valores únicos antes: ['Male' 'Female']
  Valores após encoding: [1 0]
  Mapeamento: {'Female': np.int64(0), 'Male': np.int64(1)}

Encodando 'Married'...
  Valores únicos antes: ['No' 'Yes']
  Valores após encoding: [0 1]
  Mapeamento: {'No': np.int64(0), 'Yes': np.int64(1)}

Encodando 'Education'...
  Valores únicos antes: ['Graduate' 'Not Graduate']
  Valores após encoding: [0 1]
  Mapeamento: {'Graduate': np.int64(0), 'Not Graduate': np.int64(1)}

Encodando 'Self_Employed'...
  Valores únicos antes: ['No' 'Yes']
  Valores após encoding: [0 1]
  Mapeamento: {'No': np.int64(0), 'Yes': np.int64(1)}

Encodando variável target (Loan_Status)...
  Valores únicos antes: ['Y' 'N']
  Valores após encoding: [0 1]
  Mapeamento: {'N': np.int64(0), 'Y': np.int64(1)}

✓ Encoding concluído!
  Encoders salvos: ['Gender', 'Married', 'Education'

In [11]:
# =============================================================================
# 9. ETAPA B (PARTE 2): BALANCEAMENTO COM SMOTE
# =============================================================================
print("=" * 80)
print("ETAPA B (PARTE 2): BALANCEAMENTO COM SMOTE (APENAS TREINO)")
print("=" * 80)

print("Distribuição ANTES do SMOTE:")
unique, counts = np.unique(y_train_encoded, return_counts=True)
for val, count in zip(unique, counts):
    classe_nome = le_target.inverse_transform([val])[0]
    print(f"  Classe {classe_nome} ({val}): {count} amostras ({count/len(y_train_encoded)*100:.1f}%)")
print()

# Aplicar SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train_encoded)

print("Distribuição APÓS o SMOTE:")
unique, counts = np.unique(y_train_balanced, return_counts=True)
for val, count in zip(unique, counts):
    classe_nome = le_target.inverse_transform([val])[0]
    print(f"  Classe {classe_nome} ({val}): {count} amostras ({count/len(y_train_balanced)*100:.1f}%)")
print()

print(f"✓ SMOTE aplicado!")
print(f"  Shape ANTES: {X_train.shape}")
print(f"  Shape APÓS: {X_train_balanced.shape}")
print()


ETAPA B (PARTE 2): BALANCEAMENTO COM SMOTE (APENAS TREINO)
Distribuição ANTES do SMOTE:
  Classe N (0): 154 amostras (31.4%)
  Classe Y (1): 337 amostras (68.6%)

Distribuição APÓS o SMOTE:
  Classe N (0): 337 amostras (50.0%)
  Classe Y (1): 337 amostras (50.0%)

✓ SMOTE aplicado!
  Shape ANTES: (491, 7)
  Shape APÓS: (674, 7)



In [12]:
# =============================================================================
# 10. ETAPA G: PADRONIZAÇÃO DE FEATURES NUMÉRICAS
# =============================================================================
print("=" * 80)
print("ETAPA G: PADRONIZAÇÃO DE FEATURES NUMÉRICAS (STANDARD SCALER)")
print("=" * 80)

# Identificar colunas numéricas
colunas_numericas = X_train_balanced.select_dtypes(include=[np.number]).columns.tolist()
print(f"Colunas numéricas a padronizar: {colunas_numericas}")
print()

# Aplicar StandardScaler
scaler = StandardScaler()
X_train_scaled = X_train_balanced.copy()
X_test_scaled = X_test.copy()

X_train_scaled[colunas_numericas] = scaler.fit_transform(X_train_balanced[colunas_numericas])
X_test_scaled[colunas_numericas] = scaler.transform(X_test[colunas_numericas])

print("✓ Padronização concluída!")
print(f"  Média das features (treino): {X_train_scaled[colunas_numericas].mean().mean():.6f}")
print(f"  Desvio padrão (treino): {X_train_scaled[colunas_numericas].std().mean():.6f}")
print()


ETAPA G: PADRONIZAÇÃO DE FEATURES NUMÉRICAS (STANDARD SCALER)
Colunas numéricas a padronizar: ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'ApplicantIncome', 'LoanAmount']

✓ Padronização concluída!
  Média das features (treino): -0.000000
  Desvio padrão (treino): 1.000743



In [13]:
# =============================================================================
# 11. ETAPA H: 5 PIPELINES COM GRID SEARCH E VALIDAÇÃO CRUZADA
# =============================================================================
print("=" * 80)
print("ETAPA H: CRIAÇÃO DOS 5 PIPELINES COM GRID SEARCH")
print("=" * 80)
print("⏰ Este processo pode levar alguns minutos...\n")

# Definindo os 5 modelos e seus hiperparâmetros
models_params = {
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'penalty': ['l2'],
            'solver': ['lbfgs', 'liblinear']
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'max_depth': [3, 5, 7, 10, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 150],
            'learning_rate': [0.01, 0.1, 0.2],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 1.0]
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['rbf', 'linear'],
            'gamma': ['scale', 'auto']
        }
    }
}

# Dicionário para armazenar resultados
resultados_modelos = {}

# Treinar cada modelo
import time

for nome, config in models_params.items():
    print("=" * 60)
    print(f"Treinando: {nome}")
    print("=" * 60)
    
    inicio = time.time()
    
    # Grid Search com validação cruzada (5-fold)
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        cv=5,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1
    )
    
    # Fit
    grid_search.fit(X_train_scaled, y_train_balanced)
    
    tempo = time.time() - inicio
    
    # Melhores parâmetros
    print(f"\n✓ {nome} concluído em {tempo:.1f}s")
    print(f"  Melhor score CV (AUC): {grid_search.best_score_:.4f}")
    print(f"  Melhores parâmetros: {grid_search.best_params_}")
    
    # Predições no conjunto de teste
    y_pred = grid_search.best_estimator_.predict(X_test_scaled)
    y_pred_proba = grid_search.best_estimator_.predict_proba(X_test_scaled)[:, 1]
    
    # Métricas no teste
    auc_test = roc_auc_score(y_test_encoded, y_pred_proba)
    
    print(f"\n  📊 MÉTRICAS NO CONJUNTO DE TESTE:")
    print(f"  AUC: {auc_test:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_test_encoded, y_pred, 
                                target_names=le_target.classes_))
    
    # Armazenar resultados
    resultados_modelos[nome] = {
        'grid_search': grid_search,
        'best_estimator': grid_search.best_estimator_,
        'best_params': grid_search.best_params_,
        'cv_score': grid_search.best_score_,
        'test_auc': auc_test,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba,
        'tempo': tempo
    }
    
    print()

ETAPA H: CRIAÇÃO DOS 5 PIPELINES COM GRID SEARCH
⏰ Este processo pode levar alguns minutos...

Treinando: Logistic Regression
Fitting 5 folds for each of 8 candidates, totalling 40 fits

✓ Logistic Regression concluído em 10.5s
  Melhor score CV (AUC): 0.5930
  Melhores parâmetros: {'C': 0.01, 'penalty': 'l2', 'solver': 'lbfgs'}

  📊 MÉTRICAS NO CONJUNTO DE TESTE:
  AUC: 0.6146

  Classification Report:
              precision    recall  f1-score   support

           N       0.42      0.55      0.48        38
           Y       0.77      0.66      0.71        85

    accuracy                           0.63       123
   macro avg       0.59      0.61      0.59       123
weighted avg       0.66      0.63      0.64       123


Treinando: Decision Tree
Fitting 5 folds for each of 45 candidates, totalling 225 fits

✓ Decision Tree concluído em 0.8s
  Melhor score CV (AUC): 0.6560
  Melhores parâmetros: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 10}

  📊 MÉTRICAS NO CON

In [14]:
# =============================================================================
# 12. COMPARAÇÃO DOS MODELOS E SELEÇÃO DO MELHOR
# =============================================================================
print("=" * 80)
print("COMPARAÇÃO FINAL DOS MODELOS")
print("=" * 80)
print()

# Criar DataFrame comparativo
df_comparacao = pd.DataFrame({
    'Modelo': list(resultados_modelos.keys()),
    'AUC CV (5-fold)': [r['cv_score'] for r in resultados_modelos.values()],
    'AUC Teste': [r['test_auc'] for r in resultados_modelos.values()],
    'Tempo (s)': [r['tempo'] for r in resultados_modelos.values()]
})

df_comparacao = df_comparacao.sort_values('AUC Teste', ascending=False)
df_comparacao['Ranking'] = range(1, len(df_comparacao) + 1)
df_comparacao = df_comparacao[['Ranking', 'Modelo', 'AUC CV (5-fold)', 'AUC Teste', 'Tempo (s)']]

print(df_comparacao.to_string(index=False))
print()

# Melhor modelo
melhor_modelo_nome = df_comparacao.iloc[0]['Modelo']
melhor_modelo = resultados_modelos[melhor_modelo_nome]['best_estimator']
melhor_auc = df_comparacao.iloc[0]['AUC Teste']

print(f"🏆 MELHOR MODELO: {melhor_modelo_nome}")
print(f"   AUC Teste: {melhor_auc:.4f}")
print(f"   AUC CV: {df_comparacao.iloc[0]['AUC CV (5-fold)']:.4f}")
print()

# =============================================================================
# 13. ETAPA I: SALVAMENTO DO MODELO FINAL E PREPROCESSADORES
# =============================================================================
print("=" * 80)
print("ETAPA I: SALVAMENTO DO MODELO E PREPROCESSADORES")
print("=" * 80)

# Salvar modelo (best_estimator apenas)
with open('best_loan_model.pkl', 'wb') as f:
    pickle.dump(melhor_modelo, f)
print(f"✓ Modelo salvo: best_loan_model.pkl")

# Salvar scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ Scaler salvo: scaler.pkl")

# Salvar encoders
with open('encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
print(f"✓ Encoders salvos: encoders.pkl")

# Salvar target encoder
with open('target_encoder.pkl', 'wb') as f:
    pickle.dump(le_target, f)
print(f"✓ Target encoder salvo: target_encoder.pkl")

# Salvar informações das colunas e do modelo
model_info = {
    'feature_names': X_train_scaled.columns.tolist(),
    'numeric_columns': colunas_numericas,
    'categorical_columns': list(encoders.keys()),
    'target_classes': le_target.classes_.tolist(),
    'best_model_name': melhor_modelo_nome,
    'best_auc': float(melhor_auc),
    'colunas_remover': colunas_remover
}

with open('model_info.pkl', 'wb') as f:
    pickle.dump(model_info, f)
print(f"✓ Informações do modelo salvas: model_info.pkl")

print()
print("=" * 80)
print("✅ PROCESSO COMPLETO FINALIZADO COM SUCESSO!")
print("=" * 80)
print(f"\n📊 RESUMO FINAL:")
print(f"  Melhor modelo: {melhor_modelo_nome}")
print(f"  AUC no teste: {melhor_auc:.4f}")
print(f"  Total de amostras treino (após SMOTE): {X_train_balanced.shape[0]}")
print(f"  Total de amostras teste: {X_test_scaled.shape[0]}")
print(f"  Features utilizadas: {len(model_info['feature_names'])}")
print(f"\n📁 Arquivos gerados:")
print(f"  1. best_loan_model.pkl - Modelo treinado")
print(f"  2. scaler.pkl - StandardScaler")
print(f"  3. encoders.pkl - LabelEncoders")
print(f"  4. target_encoder.pkl - Encoder do target")
print(f"  5. model_info.pkl - Metadados")
print("=" * 80)

COMPARAÇÃO FINAL DOS MODELOS

 Ranking              Modelo  AUC CV (5-fold)  AUC Teste  Tempo (s)
       1   Gradient Boosting         0.716888   0.616409  13.292411
       2       Random Forest         0.698000   0.616254  11.139475
       3 Logistic Regression         0.592958   0.614551  10.546391
       4                 SVM         0.608134   0.603715   0.855000
       5       Decision Tree         0.655979   0.553096   0.773588

🏆 MELHOR MODELO: Gradient Boosting
   AUC Teste: 0.6164
   AUC CV: 0.7169

ETAPA I: SALVAMENTO DO MODELO E PREPROCESSADORES
✓ Modelo salvo: best_loan_model.pkl
✓ Scaler salvo: scaler.pkl
✓ Encoders salvos: encoders.pkl
✓ Target encoder salvo: target_encoder.pkl
✓ Informações do modelo salvas: model_info.pkl

✅ PROCESSO COMPLETO FINALIZADO COM SUCESSO!

📊 RESUMO FINAL:
  Melhor modelo: Gradient Boosting
  AUC no teste: 0.6164
  Total de amostras treino (após SMOTE): 674
  Total de amostras teste: 123
  Features utilizadas: 7

📁 Arquivos gerados:
  1. best_